Yield Curve Fit - Nelson Siegel and Cubic Spline



In [ ]:
import pandas as pd
import numpy as np
from fredapi import Fred
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize
from scipy.interpolate import CubicSpline, UnivariateSpline
from scipy.linalg import lstsq

In [ ]:
plt.rcParams.update({
    'figure.dpi': 90,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
BLUE   = '#1f4e79'
ORANGE = '#c55a11'
GREEN  = '#375623'
GREY   = '#595959'

In [ ]:
fred = Fred(api_key='3a4a21640f653124f86acdea283a9768')

series_ids = ['DGS1MO', 'DGS3MO', 'DGS6MO', 'DGS1', 'DGS2', 'DGS3', 'DGS5', 'DGS7', 'DGS10', 'DGS20', 'DGS30']

def get_yield_data(series_id):
  data = fred.get_series(series_id, observation_start='1975-01-01', observation_end='2026-02-01')
  return data

yields_dict = {series_id : get_yield_data(series_id) for series_id in series_ids}
yields = pd.DataFrame(yields_dict)

yields.columns = ['1 Month', '3 Month', '6 Month', '1 Year', '2 Year', '3 Year', '5 Year', '7 Year', '10 Year', '20 Year', '30 Year']

In [ ]:
observation_date = '2024-09-13'
maturities = np.array([0.083, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])
bond_yields = yields.loc[observation_date]

In [ ]:
def plot_yield_curve(date):
    maturities = ['1M', '3M', '6M', '1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']
    
    fig, ax=plt.subplots(figsize=(6,4))
    ax.plot(maturities, yields.loc[date], marker='D', label='Yield Curve at ' + date)
    
    ax.set_yticklabels(['{:.2f}%'.format(y) for y in ax.get_yticks()])
    ax.set_xticks(range(len(maturities)))
    ax.set_xticklabels(maturities)

    ax.set_xlabel('Maturity')
    ax.set_ylabel('Yield')
    ax.set_title('Treasury Yield Curve')

    ax.legend(loc = [0.69, 0.04])

    plt.grid(True)
    plt.show()

In [ ]:
plot_yield_curve(observation_date)

In [ ]:
def ns_loadings(tau, lam):
  tau = np.asarray(tau, float)
  lt = tau*lam
  safe = np.where(lt==0,1e-12,lt)
  L2 = (1-np.exp(-safe))/safe
  L3 = L2 - np.exp(-safe)
  return np.ones_like(tau), L2, L3

def ns_yields(tau, b0, b1, b2, lam):
  L1, L2, L3 = ns_loadings(tau, lam)
  return b0*L1 + b1*L2 + b2*L3

def ns_forward(tau, b0, b1, b2, lam):
  exp_lt = np.exp(-lam*tau)
  return b0 + b1*exp_lt + b2*(lam*tau)*exp_lt

def fit_ns(maturities, bond_yields, n_lambda=80):
  best_sse, best_parameters= np.inf, None
  
  for lam in np.linspace(0.01, 5.0, n_lambda):
    L1, L2, L3 = ns_loadings(maturities, lam)
    X = np.column_stack([L1, L2, L3])
    betas, _, _, _ = lstsq(X,bond_yields)
    sse = float(np.sum(bond_yields - X @ betas)**2)
    if sse < best_sse:
      best_sse = sse
      best_parameters = (*betas, lam)
  b0, b1, b2, lam = best_parameters
  
  fitted = ns_yields(maturities, b0, b1, b2, lam)
  rmse = np.sqrt(np.mean((bond_yields - fitted)**2))
  mae = np.mean(np.abs(bond_yields - fitted))
  return dict(b0=b0, b1=b1, b2=b2, lam=lam, rmse=rmse, mae=mae)
NS = fit_ns(maturities, bond_yields)

In [ ]:
cs_interp = CubicSpline(maturities, bond_yields, bc_type='natural')

cs_smooth = UnivariateSpline(maturities, bond_yields, k=3, s=0.08)

def cs_forward(spline, tau, mode='cubic'):
  tau = np.asarray(tau, float)
  if mode == 'cubic':
    return spline(tau) + tau*spline(tau,1)
  else:
    return spline(tau) + tau*spline.derivative()(tau)
  
for name, pred_fn in [('Interpolating', lambda t : cs_interp(t)), ('Smoothing', lambda t : cs_smooth(t))]:
  residual = bond_yields - pred_fn(maturities)
  rmse = np.sqrt(np.mean(residual**2))
  mae = np.mean(np.abs(residual))

In [ ]:
tau_fine = np.linspace(0.25, 30, 600)
fwd_tau = np.linspace(0.5, 29.5, 500)

fig = plt.figure(figsize=(15,11))
gs = gridspec.GridSpec(2,2, hspace=0.38, wspace=0.32)

ax1=fig.add_subplot(gs[0,:])
ax1.scatter(maturities, bond_yields, color='black', zorder=6, s=70, label='Observed yields')
ax1.plot(tau_fine, ns_yields(tau_fine, **{k:NS[k] for k in ['b0','b1','b2','lam']}), color=BLUE,   lw=2.5, label=f"Nelson-Siegel  (RMSE={NS['rmse']*100:.3f}bp)")
ax1.plot(tau_fine, cs_interp(tau_fine), color=ORANGE, lw=2.2, label=f'Cubic Spline — Interpolating  (RMSE≈0bp)')
ax1.plot(tau_fine, cs_smooth(tau_fine), color=GREEN,  lw=2.0, ls='--', label=f'Cubic Spline — Smoothing')
ax1.axhline(bond_yields[2], color=GREY, lw=0.8, ls=':', alpha=0.6)
ax1.set_xlabel('Maturity (years)')
ax1.set_ylabel('Par Yield (%)')
ax1.set_title(f'US Treasury Spot Yield Curves — {observation_date}')
ax1.legend(fontsize=9)

ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(fwd_tau, ns_forward(fwd_tau, NS['b0'],NS['b1'],NS['b2'],NS['lam']),
         color=BLUE,   lw=2.5, label='NS Forward')
ax2.plot(fwd_tau, cs_forward(cs_interp, fwd_tau, 'cubic'),
         color=ORANGE, lw=2.2, label='CS Interp Forward')
ax2.plot(fwd_tau, cs_forward(cs_smooth, fwd_tau, 'smooth'),
         color=GREEN,  lw=2.0, ls='--', label='CS Smooth Forward')
ax2.scatter(maturities, bond_yields, color='black', s=40, zorder=5, alpha=0.5)
ax2.set_xlabel('Maturity (years)')
ax2.set_ylabel('Instantaneous Forward Rate (%)')
ax2.set_title('Implied Forward Rates')
ax2.legend(fontsize=9)

ax3 = fig.add_subplot(gs[1, 1])
ns_resid = (bond_yields - ns_yields(maturities, NS['b0'],NS['b1'],NS['b2'],NS['lam'])) * 100
cs_resid = (bond_yields - cs_smooth(maturities)) * 100
x = np.arange(len(maturities))
w = 0.35
ax3.bar(x - w/2, ns_resid, width=w, color=BLUE,   alpha=0.8, label='NS Residuals')
ax3.bar(x + w/2, cs_resid, width=w, color=GREEN,  alpha=0.8, label='CS Smooth Residuals')
ax3.axhline(0, color='black', lw=0.8)
ax3.set_xticks(x)
ax3.set_xticklabels([f'{m}yr' for m in maturities], rotation=45, fontsize=8)
ax3.set_ylabel('Residual (basis points)')
ax3.set_title('Fitting Residuals vs Observed Yields')
ax3.legend(fontsize=9)
plt.show()